# 1. How the process model reads an ensemble

This notebook walks through the flag-based process model of Marvel et al. (2026), as used in the TCRE
development code, on the CMIP6 land carbon feedbacks. It takes about five minutes to read and runs in
seconds, because the slow sampling has been done in advance by

```bash
python scripts/precompute.py --jobs 4
```

which caches its results in `results/precomputed/cmip6/`. Background is in `docs/guide.md`.

In [ ]:
import pandas as pd
from tcre_evidence import process_plots as pp, process_tests as pt
from tcre_evidence.paths import DATA

pd.set_option("display.precision", 2)

## The data

One value per model of the land carbon-climate feedback γ_L (PgC/K, COU−BGC convention), and one 0/1 flag
per process saying whether the model represents it. The `family` column groups models that share code.

In [ ]:
gamma, flags, family = pt.load_ensemble(DATA / "examples" / "cmip6_land_feedbacks.csv",
                                        DATA / "examples" / "cmip6_process_flags.csv", "gamma_land_coubgc")
flags.assign(gamma_land=gamma.round(1), family=family)

## The question the model asks

Do models with a process have different values from models without it? Each panel splits the ensemble by
one flag; the bars are the group means. Note how few models carry some flags, and that the three
permafrost models all belong to one family.

In [ ]:
pp.by_flag(gamma, flags, family, "γ_L (PgC/K)");

## The process model

The model explains each model's value as a baseline plus the effect of each process it has, plus errors:

    γ_L of model m = U + Σ_p flag_mp × δ_p + shared error + model error

For β_L the effects multiply instead (factors η_p). Its estimate for the **real world** is the baseline with
**every** process switched on at its average effect, U + Σ δ_p, and without the errors.

Each term starts from a prior (the development code's own). The figure compares that prior (grey) with the
posterior after seeing the 13 models (blue). If they look the same, the ensemble has taught the model
nothing about that term, and the term's value in the result is the prior's.

In [ ]:
res_gamma = pp.load("cmip6", "gamma_land_coubgc")
pp.prior_posterior(res_gamma);

`width_ratio` is the width of the posterior 90 % interval over the prior's: near 1 means not informed.
A narrower posterior is not proof that a term is learned, though. The permafrost prior has a long tail, and
the data can rule out that tail without saying anything about the effect itself. Notebook 2 tests this
directly.

In [ ]:
res_gamma["prior_vs_posterior"].assign(term=lambda t: t.term.map(pp.pretty))

## The same for β_L

The land carbon-concentration feedback (PgC/ppm) uses the multiplicative form: each process scales the
baseline by a factor η. A factor of 1 means no effect.

In [ ]:
beta, _, _ = pt.load_ensemble(DATA / "examples" / "cmip6_land_feedbacks.csv",
                              DATA / "examples" / "cmip6_process_flags.csv", "beta_land_coubgc")
pp.by_flag(beta, flags, family, "β_L (PgC/ppm)");

In [ ]:
res_beta = pp.load("cmip6", "beta_land_coubgc")
pp.prior_posterior(res_beta);

## What to take from this

* The process model asks a simple question of the ensemble: do flagged and unflagged models differ?
* With about 13 models, a few flags per process and a large spread within each group, that difference is
  hard to measure, and some terms leave the data with roughly the prior they started with.
* Notebook 2 asks directly which effects this ensemble can measure.